In [ ]:
import pandas as pd
import numpy as np

rentals = pd.read_csv('data/bike_rentals_clean.csv', parse_dates=['start_date', 'end_date'])
rentals.shape

## Lab 5 - Part 2
### Ex 1

In [ ]:
#a)
print('n_median-price_per_station', rentals[['price', 'station']].groupby('station')
    .agg(n=('price', 'count'), median=('price', 'median')).sort_values('median', ascending=False))

#b)
print('\nn_median-price_per_station_ebikes', rentals.loc[(rentals['bike_type'] == 'E-bike'),['price','station','bike_type']].groupby('station')
    .agg(n=('price', 'count'), median=('price', 'median')).sort_values('median', ascending=False))
print(1554/1220)

#c)
print('\n', (rentals['bike_type'] == 'E-bike').groupby(rentals['station']).mean().sort_values(ascending=False).round(2) * 100)

#### (a) Kungsholmen 73 rentals, median 742, Södermalm 87, 720; Östermalm 48, 710;Hammarby 47, 609; Central 95, 596. 
#### (b) E-bikes: Kungsholmen 22, 1554; Södermalm 25, 1220, Central 27, 1144; Östermalm 19, 1088; Hammarby 15, 712. 
#### (c) Share of e-bikes: Östermalm 39, Hammarby 31, Södermalm 31, Kungsholmen 30, Central 28 (percent).
#### Share of E-bikes quite even across stations. Kungsholmen highest median for both groups.
#### Price disparity not that great in the overall grouping. In the E-bike case the price dif between Kungsholmen and second place and the competition is >27% which is significantly more expensive. We don't have any data on vehicle quality justifying the higher price.
### Ex 2

In [ ]:
false_rental_period = rentals.loc[rentals['rental_days'] < 0, ['rental_id', 'rental_days']]
print(false_rental_period)
print(rentals['rental_days'].value_counts(dropna=False))

rentals['rental_days'] = rentals['rental_days'].where(rentals['rental_days'] >= 0, np.nan)
print(rentals['rental_days'].value_counts(dropna=False))

#### invalid loan period: 6      R-5077         -1.0 , 196    R-5217         -2.0
#### 4 NaN after where(). 2 invalid just swapped and 2 previous impossible dates.
#### I would ask if the dates are manually entered or schema automated.
### Ex 3

In [ ]:
rentals_raw = pd.read_csv('data/bike_rentals.csv')

def the_whole_shabang(raw):
    #drop empty columns
    rentals = raw.dropna(axis=1, how='all')
    #remove chars and cast price to float
    rentals['price'] = (rentals['price']
                    .str.strip()
                    .str.replace('SEK', '')
                    .str.replace(',', '')
                    .astype(float))
    # setting all t,f to True False
    t_f_col = [c for c in rentals.columns if set(rentals[c].dropna().unique()) <= {'t', 'f'}]
    for c in t_f_col:
        rentals[c] = rentals[c].map({'t': True, 'f': False})
    # casting date columns to datetime
    date_cols = ['start_date','end_date']
    for c in date_cols:
        rentals[c] = pd.to_datetime(rentals[c], errors='coerce')
    # adding rental_days column
    rentals['rental_days'] = (rentals['end_date'] - rentals['start_date']).dt.days
    # adding has_rating column
    rentals['has_rating'] = rentals['rating'].notna()
    # adding child_helmet, helmet_counts, settinng helmet_counts 0 for 'No helmets'
    rentals['child_helmet'] = rentals['helmets_text'].str.contains('child')
    rentals['helmets_count'] = rentals['helmets_text'].str.extract(r'(\d+)', expand=False)
    rentals['helmets_count'] = rentals['helmets_count'].where(~rentals['helmets_text'].str.contains('No helmets'), 0)
    rentals['helmets_count'] = rentals['helmets_count'].astype(float)
    # fixing station variation duplicates
    rentals['station'] = rentals['station'].str.strip().str.lower().str.title()
    # drop duplicates
    rentals = rentals.drop_duplicates().reset_index(drop=True)
    # NaN on invalid date period
    rentals['rental_days'] = rentals['rental_days'].where(rentals['rental_days'] >= 0, np.nan)
    return rentals

rentals_one_shot = the_whole_shabang(rentals_raw)
print(rentals.shape)
print(rentals_one_shot.shape)
rentals_one_shot.dtypes

#### rentals_one_shot shape: (400, 16) seems legit on-god fr fr no kap.
#### types of concern: insured: object. because bool field can not include null values.
### Ex 4
####
#### Issues:
#### Dropped 3 empty columns. Remove.
#### Price cleaned and converted to number. Changed.
#### Missing prices left as are. Most because of Cancelled booking. Leave.
#### Impossible dates and negative period delta turned into NaT. Changed.
#### Added column has_rating bool for if('rating'). Fill.
#### Added column child_helmet bool for if('helmets_text'.contains('child')). Fill.
#### Added column helmets_count based on helmets_text numbers and 0 for 'No'. Fill.
#### Cleaned duplicate variation of station. Changed.
#### Removed 3 true duplicate entries. Remove.
#### Set 2 negative period delta rental_days to NaN. Changed.